# 01 - Data Preparation
**EL SHARNOUBY GROUP - HVIA project.** Business understanding, data understanding, data quality, integration and the KPI framework.

**Flow:** raw CSV -> validation -> cleaning -> standardisation -> integration -> analytical datasets (`data/processed/`). Raw files are never modified.

## 1. Business understanding
EL SHARNOUBY GROUP is an Alexandria-based apparel group. Three business units matter for this analysis:
- **LEO MODA** - children's/family apparel brand sold through a retail store (Sidi Bishr) and an online store.
- **El-Sharnouby Wholesale Desk** - B2B sales of the same products to resellers (the volume engine).
- **LEO PRODUCTION HUB / LEO PRINT** - the factory (cutting, sewing, printing, finishing, packing) and the printing house.

Flow of value: *suppliers -> materials -> production lines -> printing -> QC -> inventory -> three sales channels -> customers -> returns/service*. Marketing and web traffic feed the online channel.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.float_format", "{:,.3f}".format)
import analysis as A, modeling as M, visuals as V
from utils import fmt_egp, fmt_pct, fmt_int

In [2]:
units = A.pd.read_csv(ROOT / "data" / "raw" / "Business_Units.csv", encoding="utf-8-sig")
units.drop(columns=[c for c in units.columns if "Source" in c or "Class" in c], errors="ignore")

,Business_Unit,Function,Public_Capacity
0,LEO DESIGN,"Fashion design, pattern making, 3D visualization",Not publicly stated
1,LEO TEX,Woven fabric manufacturing,Integrated textile manufacturing; exact capaci...
2,LEO PRODUCTION HUB,Garment manufacturing,"Up to 5,000 pieces/day"
3,LEO PRINT,Textile and garment printing,"Up to 20,000 garments/day and 6,000 fabric met..."
4,LEO MODA,"Retail, online and wholesale apparel brand",Delivery across Egypt; retail + wholesale chan...


## 2. Data understanding - what exists, at what grain

In [3]:
import pandas as pd
rows = []
for name in ["Sales_Transactions","Customers","Product_Master","Locations","Inventory_History","Marketing","Web_Traffic_Daily","Materials","Product_BOM","Suppliers","Purchase_Orders","Production_Orders","Printing_Batches","Machine_Downtime","Quality","Customer_Service","Group_Operations","Business_Units"]:
    df = pd.read_csv(ROOT / "data" / "raw" / f"{name}.csv", encoding="utf-8-sig")
    rows.append((name, len(df), df.shape[1], df.iloc[:, 0].name))
grain = pd.DataFrame(rows, columns=["Table", "Rows", "Columns", "Key column"])
grain["Grain (one row per...)"] = [
 "invoice line (one product per invoice)", "customer", "product (59 SKUs)", "location (3)", "month x location x product",
 "campaign", "day", "material (12)", "product x material", "supplier (12)", "purchase order", "production order",
 "print batch", "downtime event", "QC inspection", "service ticket", "month x business unit", "business unit"]
grain

,Table,Rows,Columns,Key column,Grain (one row per...)
0,Sales_Transactions,180000,30,Row_ID,invoice line (one product per invoice)
1,Customers,30000,9,Customer_ID,customer
2,Product_Master,59,12,Product_ID,product (59 SKUs)
3,Locations,5,9,Location_ID,location (3)
4,Inventory_History,6372,19,Snapshot_Month,month x location x product
5,Marketing,420,15,Campaign_ID,campaign
6,Web_Traffic_Daily,1096,11,Date,day
7,Materials,12,6,Material_ID,material (12)
8,Product_BOM,337,7,Product_ID,product x material
9,Suppliers,12,7,Supplier_ID,supplier (12)


**Relationships** (all joins are many-to-one, validated with `validate="many_to_one"` and row-count checks in `checked_merge`):

`Sales -> Customers, Product_Master, Locations` | `Inventory -> Product_Master, Locations (reconciles exactly to Sales units)` | `Marketing -> Product_Master` | `Purchase_Orders -> Suppliers, Materials` | `Product_BOM -> Product_Master, Materials` | `Production/Quality -> Product_Master` | `Printing/Downtime -> Machine_ID`.

Web traffic, customer service and group operations are time-series tables joined on month only.

## 3. Run the pipeline and review data quality

In [4]:
import data_pipeline
counts = data_pipeline.run()
pd.Series(counts, name="rows").to_frame()

[pipeline] 19 processed datasets written | 81 checks | 10 flagged


,rows
sales,180000
customer_360,25634
demand_monthly,6353
product_quality_returns,59
inventory,6372
procurement,3600
production,2500
printing,1900
downtime,2800
quality,9000


In [5]:
dq = pd.read_csv(ROOT / "outputs" / "tables" / "data_quality_report.csv")
print(f"{len(dq)} checks run, {(dq.status=='FLAG').sum()} flagged")
dq[dq.status == "FLAG"][["table", "check", "rows_failed", "pct_failed", "action"]]

81 checks run, 10 flagged


,table,check,rows_failed,pct_failed,action
48,Sales_Transactions,Negative gross profit (loss-making invoice),4876,2.709,kept & flagged: business finding
49,Sales_Transactions,Customer type inconsistent with channel,2197,1.221,kept & flagged
54,Inventory_History,stock roll-forward does not balance,147,2.307,kept; gap stored as Unexplained_Stock_Adj
60,Marketing,Attributed revenue = 5.5x total Online net sal...,1,100.000,"ROAS used for ranking only, not as absolute"
62,Web_Traffic_Daily,Web orders < Online invoices in Sales,1,100.000,flagged: web analytics under-counts or Sales o...
67,Purchase_Orders,Supplier category differs from material category,2685,74.583,kept: suppliers are not category-specialised
74,Printing_Batches,Printed > Planned (over-run),367,19.316,kept & flagged
75,Printing_Batches,Rejected units without Defect_Type,381,20.053,labelled 'Unclassified'
78,Quality,Rejected units without Defect_Type,1621,18.011,labelled 'Unclassified'
79,Quality,Rejects with root cause 'No significant issue',1612,17.911,kept & flagged: root-cause coding unreliable


**What the checks tell us**
- Structurally the data is clean: no duplicates, no orphan keys, no invalid dates, all arithmetic identities hold (gross = qty x price, net = gross x (1-discount), GP = net - COGS).
- **Business-meaningful flags (kept, not deleted):** 4,876 loss-making wholesale invoices; 2,197 invoices where customer type contradicts the channel; 147 inventory rows that do not roll forward.
- **Reconciliation gaps to respect in the analysis:** marketing *attributed* revenue is 5.5x total online net sales (so ROAS is used to **rank** platforms, not as absolute truth); web-analytics orders (47.5K) are fewer than online invoices (71.6K); QC and print rejects with no defect label are tagged `Unclassified` (12% of QC loss); suppliers are not specialised by material category.
- Production orders (6.3M units) are 3.8x units sold (1.66M): the hub clearly produces for other brands/OEM or builds stock, so **sales-to-production linkage is analysed through rates, not volumes**.

## 4. Integration checks - row counts before and after joins

In [6]:
d = A.load_all()
s = d["sales"]
raw_sales = pd.read_csv(ROOT / "data" / "raw" / "Sales_Transactions.csv", encoding="utf-8-sig")
c360 = d["customer_360"]
checks = pd.DataFrame({
  "Check": ["Sales rows: raw vs processed (enriched)", "Customers with sales vs Customer_360 rows", "Inventory units sold vs Sales units", "Net sales: raw vs processed"],
  "Left": [len(raw_sales), s.Customer_ID.nunique(), d["inventory"].Sold_Qty.sum(), raw_sales.Net_Sales_EGP.sum()],
  "Right": [len(s), len(c360), s.Quantity.sum(), s.Net_Sales_EGP.sum()]})
checks["Match"] = (checks.Left - checks.Right).abs() < 1
checks

,Check,Left,Right,Match
0,Sales rows: raw vs processed (enriched),"180,000.000","180,000.000",True
1,Customers with sales vs Customer_360 rows,"25,634.000","25,634.000",True
2,Inventory units sold vs Sales units,"1,660,589.000","1,660,589.000",True
3,Net sales: raw vs processed,"421,338,377.530","421,338,377.530",True


## 5. KPI framework
Only KPIs the data supports. Everything downstream uses these definitions.

In [7]:
A.KPI_DEFINITIONS

,KPI,Definition,Source
0,Net Sales,"Sum of Net_Sales_EGP (after discount, before r...",Sales_Transactions
1,Gross Profit,Net sales - estimated COGS,Sales_Transactions
2,Gross Margin %,Gross profit / Net sales,Sales_Transactions
3,Avg Discount %,1 - Net sales / Gross sales (value-weighted),Sales_Transactions
4,AOV,Net sales / invoices,Sales_Transactions
5,Refund Rate,Refund value / Net sales,Sales_Transactions
6,Repeat Rate,Customers with >1 order / customers with >=1 o...,Customer_360
7,ROAS,Attributed revenue / Spend (platform-reported),Marketing
8,CAC,Spend / attributed orders,Marketing
9,Web Conversion,Orders / Sessions,Web_Traffic_Daily


In [8]:
total = A.kpi_total(s)
pd.Series({"Net sales": fmt_egp(total["Net_Sales"]), "Gross profit": fmt_egp(total["Gross_Profit"]), "Gross margin": fmt_pct(total["Gross_Margin"]),
           "Units": fmt_int(total["Units"]), "Invoices": fmt_int(total["Invoices"]), "Customers": fmt_int(total["Customers"]), "Avg discount": fmt_pct(total["Avg_Discount"]), "Refund rate": fmt_pct(total["Refund_Rate"])}, name="Group KPIs (Oct-2023 .. Sep-2026)").to_frame()

,Group KPIs (Oct-2023 .. Sep-2026)
Net sales,EGP 421.3M
Gross profit,EGP 64.9M
Gross margin,15.4%
Units,"1,660,589"
Invoices,"180,000"
Customers,"25,634"
Avg discount,29.8%
Refund rate,2.2%


## Conclusions
1. Data is structurally reliable; every number downstream ties back to raw totals.
2. The data-quality story is in **business flags** (loss-making invoices, channel/customer-type conflicts) and **reconciliation gaps** (marketing attribution, web orders, production volume), all documented and handled explicitly.
3. Processed datasets are ready: `sales`, `customer_360`, `inventory`, `procurement`, `production`, `printing`, `quality`, `marketing`, ...